# Play Tic-Tac-Toe Interactively (Simple Implementation)

To play interactively, you need to run the whole notebook in JupyterLab and scroll down.

The code for the basic functions used for search are implemented in [tictactoe.py](tictactoe.py).

In [46]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("tictactoe.py", ".")

In [47]:
from tictactoe import empty_board, actions, result, terminal, utility, other, show_board
from tictactoe import random_player, play 

## Implement an Interactive Game


### Interactive Player

Implement the interactive player. It shows the board and asks you for the move.

In [48]:
from IPython.display import clear_output

def interactive_player(board, player = None):
    clear_output(wait = False)
    show_board(board, help = True)
    
    available = actions(board)
    print(f'Available actions are: {available}')

    retry = True
    while retry:
        try:
            move = int(input("Your move (0-8 or -1 to end the game):\n"))            
        
            if move in available:
                retry = False
            else:
                raise ValueError()
        except ValueError:
            if move == -1:
                print("Game ended by user.")
                raise Exception("Game ended by user.")
            
            print("Please enter a valid move.")
            

    return(move)

In [ ]:
import numpy as np, math, builtins
np.random.seed(7)

# Các tác tử đã xây dựng ở các bài trước (code lấy nguyên từ các notebook, gom trong agents_ttt.py)
from agents_ttt import make_ucb1
ucb1_100_player = make_ucb1(100)                                         # UCB1/MCTS, N = 100 playout / nước đi

## Start an interactive game as x (Tự chơi)

In [50]:
# play(interactive_player, random_player, N = 1,  show_final_board=True)

### Start an interactive game as o

In [51]:
# play(random_player, interactive_player, N = 1, show_final_board = True)

## Ghi lại từng nước đi của các ván
Hàm `play_logged` in bàn cờ dạng chữ sau mỗi nước, ai đi, và **giá trị minimax** của nước vừa chọn (+1: X thắng chắc, 0: hòa, −1: O thắng chắc – tính từ góc nhìn X).

In [52]:
from tictactoe import check_board, result, empty_board
import tictactoe

def board_text(b):
    return "\n".join(" " + " | ".join(b[r*3+c] if b[r*3+c] != ' ' else str(r*3+c) for c in range(3)) for r in range(3))

# trong lúc ghi ván, không xóa màn hình và dùng bàn cờ dạng chữ để giữ lại toàn bộ lịch sử
_clear_orig, _show_orig = clear_output, show_board
clear_output = lambda **k: None
show_board = lambda b, help=True: None

def play_logged(x_agent, o_agent, x_name, o_name, seed=0):
    np.random.seed(seed)
    b = empty_board(); turn = 'x'; n = 0
    print(f"=== {x_name} (X)  vs  {o_name} (O) ===")
    while True:
        agent, name = (x_agent, x_name) if turn == 'x' else (o_agent, o_name)
        a = agent(b, turn)
        b = result(b, turn, a); n += 1
        note = ""
        if check_board(b) == 'n':
            nxt = other(turn)                       # người đi tiếp theo
            v = _max_ab(b, nxt, -math.inf, math.inf, 0, None, eval_exact)[0]   # giá trị từ góc nhìn người đi tiếp
            v_x = v if nxt == 'x' else -v           # đổi về góc nhìn của X
            note = f"  [giá trị minimax của thế cờ, góc nhìn X: {v_x:+d}]"
        print(f"Nước {n}: {turn.upper()} ({name}) đánh ô {a}{note}")
        print(board_text(b)); print()
        w = check_board(b)
        if w != 'n':
            print("KẾT QUẢ:", {"x": "X thắng", "o": "O thắng", "d": "Hòa"}[w]); print(); return w
        turn = other(turn)

### Bạn (X, cẩn thận) vs UCB1 N=100 (O)

In [53]:
play_logged(interactive_player, ucb1_100_player, 'Bạn (sensible)', 'UCB1(100)', seed=3)

=== Bạn (sensible) (X)  vs  UCB1(100) (O) ===
Available actions are: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Nước 1: X (Bạn (sensible)) đánh ô 0  [giá trị minimax của thế cờ, góc nhìn X: +0]
 x | 1 | 2
 3 | 4 | 5
 6 | 7 | 8

Nước 2: O (UCB1(100)) đánh ô 1  [giá trị minimax của thế cờ, góc nhìn X: +1]
 x | o | 2
 3 | 4 | 5
 6 | 7 | 8

Available actions are: [2, 3, 4, 5, 6, 7, 8]
Nước 3: X (Bạn (sensible)) đánh ô 4  [giá trị minimax của thế cờ, góc nhìn X: +1]
 x | o | 2
 3 | x | 5
 6 | 7 | 8

Nước 4: O (UCB1(100)) đánh ô 8  [giá trị minimax của thế cờ, góc nhìn X: +1]
 x | o | 2
 3 | x | 5
 6 | 7 | o

Available actions are: [2, 3, 5, 6, 7]
Nước 5: X (Bạn (sensible)) đánh ô 3  [giá trị minimax của thế cờ, góc nhìn X: +1]
 x | o | 2
 x | x | 5
 6 | 7 | o

Nước 6: O (UCB1(100)) đánh ô 5  [giá trị minimax của thế cờ, góc nhìn X: +1]
 x | o | 2
 x | x | o
 6 | 7 | o

Available actions are: [2, 6, 7]
Nước 7: X (Bạn (sensible)) đánh ô 6
 x | o | 2
 x | x | o
 x | 7 | o

KẾT QUẢ: X thắng



np.str_('x')

### UCB1 N=100 (X) vs Bạn (O, cẩn thận)

In [55]:
play_logged(ucb1_100_player, interactive_player, 'UCB1(100)', 'Bạn (sensible)', seed=3)

=== UCB1(100) (X)  vs  Bạn (sensible) (O) ===
Nước 1: X (UCB1(100)) đánh ô 2  [giá trị minimax của thế cờ, góc nhìn X: +0]
 0 | 1 | x
 3 | 4 | 5
 6 | 7 | 8

Available actions are: [0, 1, 3, 4, 5, 6, 7, 8]
Nước 2: O (Bạn (sensible)) đánh ô 4  [giá trị minimax của thế cờ, góc nhìn X: +0]
 0 | 1 | x
 3 | o | 5
 6 | 7 | 8

Nước 3: X (UCB1(100)) đánh ô 8  [giá trị minimax của thế cờ, góc nhìn X: +0]
 0 | 1 | x
 3 | o | 5
 6 | 7 | x

Available actions are: [0, 1, 3, 5, 6, 7]
Nước 4: O (Bạn (sensible)) đánh ô 5  [giá trị minimax của thế cờ, góc nhìn X: +0]
 0 | 1 | x
 3 | o | o
 6 | 7 | x

Nước 5: X (UCB1(100)) đánh ô 3  [giá trị minimax của thế cờ, góc nhìn X: +0]
 0 | 1 | x
 x | o | o
 6 | 7 | x

Available actions are: [0, 1, 6, 7]
Nước 6: O (Bạn (sensible)) đánh ô 1  [giá trị minimax của thế cờ, góc nhìn X: +0]
 0 | o | x
 x | o | o
 6 | 7 | x

Nước 7: X (UCB1(100)) đánh ô 7  [giá trị minimax của thế cờ, góc nhìn X: +0]
 0 | o | x
 x | o | o
 6 | x | x

Available actions are: [0, 6]
Nước 8

'd'